# EPO MinerU RAG with Reranker

In [2]:
# =========================
# 1. Install dependencies
# =========================
!pip install -q \
   langchain \
   langchain-community \
   langchain-huggingface \
   sentence-transformers \
   faiss-cpu \
   rank_bm25 \
   transformers \
   accelerate \
   openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
# =========================
# 2. Shared imports
# =========================
from pathlib import Path
from itertools import islice
from typing import Any, Dict, List, Optional, Sequence
import hashlib
import json
import os
import shutil

import torch
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder
from openai import OpenAI
from IPython.display import display, HTML, Markdown

print("Setup ready.")


/tmp/ipykernel_1082/2978977502.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Setup ready.


In [5]:
# =========================
# Build / load embeddings + BM25 from the ingest folder
# =========================
INGESTED_DATA_DIR = Path(globals().get("OUTPUT_DIR", "ingested_data"))
INDEX_NAME = INGESTED_DATA_DIR.name
VECTOR_STORE_DIR = Path("vector_stores")
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
REBUILD_VECTOR_STORE = True
BM25_CANDIDATES = 80
VECTOR_CANDIDATES = 80

vector_store_path = VECTOR_STORE_DIR / f"{INDEX_NAME}_faiss_reranker"
index_meta_path = vector_store_path / "index_meta.json"


def discover_chunk_paths(ingested_dir: Path) -> List[Path]:
    chunks_dir = ingested_dir / "chunks"
    all_chunks = chunks_dir / "all_chunks.jsonl"
    if all_chunks.exists():
        return [all_chunks]
    paths = sorted(path for path in chunks_dir.glob("*.jsonl") if path.name != "all_chunks.jsonl")
    if not paths:
        raise FileNotFoundError(f"No chunk JSONL files found under {chunks_dir}")
    return paths


def fingerprint_files(paths: Sequence[Path]) -> str:
    digest = hashlib.sha1()
    for path in paths:
        digest.update(path.as_posix().encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def load_documents_from_chunks(ingested_dir: Path) -> tuple[List[Document], List[Path], str]:
    chunk_paths = discover_chunk_paths(ingested_dir)
    fingerprint = fingerprint_files(chunk_paths)
    documents: List[Document] = []

    for chunk_path in chunk_paths:
        with chunk_path.open("r", encoding="utf-8") as file:
            for line in file:
                if not line.strip():
                    continue

                chunk = json.loads(line)
                text = (chunk.get("text") or chunk.get("content") or "").strip()

                html_meta = dict(chunk.get("metadata") or {})
                chunk_type = html_meta.get("chunk_type")

                if chunk_type == "table" and html_meta.get("table_markdown_path"):
                    md_path = Path(html_meta["table_markdown_path"])
                    if md_path.exists():
                        text = md_path.read_text(encoding="utf-8").strip()

                if not text:
                    continue

                metadata = {
                    "chunk_id": chunk.get("id"),
                    "chunk_file": str(chunk_path),
                    "index_name": INDEX_NAME,
                    "chunk_type": chunk_type,
                    "html_tag": html_meta.get("html_tag", ""),
                    "css_color": html_meta.get("styling", {}).get("color", ""),
                    "css_font": html_meta.get("styling", {}).get("fontFamily", ""),
                    "css_background": html_meta.get("styling", {}).get("backgroundColor", ""),
                    "styling_raw": json.dumps(html_meta.get("styling", {})),
                }

                documents.append(Document(page_content=text, metadata=metadata))

    if not documents:
        raise ValueError(f"Chunk files were found under {ingested_dir}, but no text chunks were loaded.")
    return documents, chunk_paths, fingerprint


def load_index_meta(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


documents, chunk_paths, chunk_fingerprint = load_documents_from_chunks(INGESTED_DATA_DIR)
current_index_meta = {
    "index_name": INDEX_NAME,
    "ingested_data_dir": str(INGESTED_DATA_DIR),
    "chunk_paths": [str(path) for path in chunk_paths],
    "chunk_count": len(documents),
    "chunk_fingerprint": chunk_fingerprint,
    "embedding_model": EMBEDDING_MODEL,
}

print(f"Corpus: {INDEX_NAME}")
print(f"Loaded chunks: {len(documents):,}")
print(f"Chunk files: {[str(path) for path in chunk_paths]}")
print(f"Loading embeddings: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

existing_index_meta = load_index_meta(index_meta_path)
index_is_current = (
    vector_store_path.exists()
    and existing_index_meta.get("chunk_fingerprint") == chunk_fingerprint
    and existing_index_meta.get("embedding_model") == EMBEDDING_MODEL
)

if index_is_current and not REBUILD_VECTOR_STORE:
    print(f"Loading existing FAISS index: {vector_store_path}")
    vector_store = FAISS.load_local(
        str(vector_store_path),
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    print(f"Building FAISS index: {vector_store_path}")
    vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)
    vector_store.save_local(str(vector_store_path))
    index_meta_path.write_text(json.dumps(current_index_meta, indent=2), encoding="utf-8")

print("Building BM25 retriever from the same chunks.")
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = BM25_CANDIDATES

print(f"Loading reranker: {RERANKER_MODEL}")
reranker_device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    cross_encoder = CrossEncoder(
        RERANKER_MODEL,
        model_kwargs={"torch_dtype": "auto"},
        trust_remote_code=True,
        device=reranker_device,
    )
except TypeError:
    cross_encoder = CrossEncoder(
        RERANKER_MODEL,
        automodel_args={"torch_dtype": "auto"},
        trust_remote_code=True,
        device=reranker_device,
    )


def _doc_key(doc: Document) -> str:
    return str(doc.metadata.get("chunk_id") or hashlib.sha1(doc.page_content.encode("utf-8")).hexdigest())


def retrieve_and_rerank(
    query: str,
    top_k: int = 14,
    candidates_k: int = 80,
    alpha: float = 0.55,
) -> List[Document]:
    candidates_k = min(candidates_k, len(documents))
    if candidates_k <= 0:
        return []

    vector_docs = vector_store.similarity_search(query, k=candidates_k)
    bm25_retriever.k = candidates_k
    bm25_docs = bm25_retriever.invoke(query)

    fused: Dict[str, Dict[str, Any]] = {}

    def add_score(found_docs: Sequence[Document], weight: float) -> None:
        for rank, doc in enumerate(found_docs):
            key = _doc_key(doc)
            if key not in fused:
                fused[key] = {"doc": doc, "score": 0.0}
            fused[key]["score"] += weight * (1.0 / (rank + 60))

    add_score(vector_docs, alpha)
    add_score(bm25_docs, 1 - alpha)

    candidates = [item["doc"] for item in sorted(fused.values(), key=lambda item: item["score"], reverse=True)]
    candidates = candidates[:candidates_k]
    if not candidates:
        return []

    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc.metadata["rerank_score"] = float(score)

    return sorted(candidates, key=lambda doc: doc.metadata.get("rerank_score", 0.0), reverse=True)[:top_k]

print("Retrieval indexes are ready.")
print(f"FAISS path: {vector_store_path}")


Corpus: ingested_data
Loaded chunks: 103
Chunk files: ['/content/drive/MyDrive/Colab Notebooks/RAG/ingested_data/chunks/all_chunks.jsonl']
Loading embeddings: sentence-transformers/all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Retrieval indexes are ready.
FAISS path: /content/drive/MyDrive/Colab Notebooks/RAG/vector_stores/ingested_data_faiss_reranker


In [8]:
# =========================
# EPO / Interinstitutional Style Guide Q&A
# =========================
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
client = OpenAI(api_key=OPENAI_API_KEY)


APPLY_HTML_STYLING = True

ANSWER_SYSTEM_PROMPT = """You are a careful retrieval-grounded assistant for the EPO corpus in this notebook.

Your job is to answer the user's actual question using the retrieved context first. Be practical, precise, and helpful. Use exact rules, names, numbers, examples, and exceptions when the context supports them. Do not invent details. If the context supports only a partial answer, give the useful part and clearly say what is not covered.

Write in the user's language unless the user asks for another language. Use concise headings or bullets when they make the answer easier to read. Do not over-explain the retrieval process."""

CONTEXT_CHECK_SYSTEM_PROMPT = """You judge whether retrieved passages are sufficient to answer a user's question.

Be pragmatic: sufficient means the assistant can give a useful, responsible answer, not that every possible detail has been retrieved. Mark the context insufficient only when a missing rule, definition, table, exception, or example would materially change the answer.

If more retrieval is needed, produce one targeted search query for the missing information. The new_query must be a retrieval query, not a request for the user to clarify. Avoid broad restatements of the original question.

Return only JSON with keys: sufficient, reason, new_query."""

def generate_answer_with_gpt(prompt: str) -> str:
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )
        return response.choices[0].message.content
    except Exception as exc:
        return f"OpenAI request failed: {exc}"

def check_if_context_sufficient(query: str, context: str) -> Dict[str, Any]:
    check_prompt = f"""Question:
{query}

Retrieved context:
{context}

Decide whether the context is enough for a useful answer. If it is not enough, write one focused retrieval query.

Return JSON only:
{{
  "sufficient": true,
  "reason": "short explanation",
  "new_query": ""
}}"""

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": CONTEXT_CHECK_SYSTEM_PROMPT},
                {"role": "user", "content": check_prompt},
            ],
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)
    except Exception as exc:
        print(f"Context check failed, continuing with the retrieved context: {exc}")
        return {"sufficient": True, "reason": "Context check failed; using retrieved context.", "new_query": ""}

def source_label(doc: Document, index: int) -> str:
    meta = doc.metadata
    source = meta.get("source_pdf") or meta.get("source_html") or meta.get("chunk_file") or "source"
    chunk_id = meta.get("chunk_id", f"chunk-{index}")
    chunk_type = meta.get("chunk_type", "text")
    return f"Source {index} | {chunk_id} | {source} | {chunk_type}"

def build_context(docs: Sequence[Document], max_chars: int = 60000) -> str:
    sections: List[str] = []
    used_chars = 0
    for index, doc in enumerate(docs, start=1):
        header = source_label(doc, index)
        text = doc.page_content.strip()

        meta = doc.metadata
        style_info = ""
        if meta.get("html_tag") or meta.get("css_color"):
            tag = meta.get("html_tag", "span")
            color = meta.get("css_color", "")
            font = meta.get("css_font", "")
            bg = meta.get("css_background", "")
            style_info = f" [HTML_TAG: {tag} | CSS_COLOR: {color} | CSS_FONT: {font} | CSS_BG: {bg}]"

        header_with_style = f"[{header}{style_info}]"

        remaining = max_chars - used_chars - len(header_with_style) - 8
        if remaining <= 0:
            break
        if len(text) > remaining:
            text = text[:remaining].rstrip() + "..."

        section = f"{header_with_style}\n{text}"
        sections.append(section)
        used_chars += len(section)
    return "\n\n".join(sections)

def unique_docs(existing: Sequence[Document], new_docs: Sequence[Document]) -> List[Document]:
    output: List[Document] = []
    seen = set()
    for doc in list(existing) + list(new_docs):
        key = doc.metadata.get("chunk_id") or doc.page_content
        if key in seen:
            continue
        output.append(doc)
        seen.add(key)
    return output

def rag_answer(
    query: str,
    max_iterations: int = 3,
    top_k: int = 14,
    candidates_k: int = 80,
    context_char_limit: int = 60000,
):
    all_docs: List[Document] = []
    current_query = query
    context = ""

    print("\n" + "=" * 60)
    print(f"Question: {query}")
    print("=" * 60)

    for iteration in range(max_iterations):
        print(f"Iteration {iteration + 1}/{max_iterations}")
        print(f"Search query: {current_query}")

        docs = retrieve_and_rerank(current_query, top_k=top_k, candidates_k=candidates_k)
        all_docs = unique_docs(all_docs, docs)
        context = build_context(all_docs, max_chars=context_char_limit)
        print(f"Retrieved unique chunks: {len(all_docs)} | context chars: {len(context):,}")

        check_result = check_if_context_sufficient(query, context)
        print(f"Context check: {check_result.get('reason', 'No reason returned')}")

        if check_result.get("sufficient", False):
            print("Context is sufficient. Writing the answer.\n")
            break

        new_query = (check_result.get("new_query") or "").strip()
        if not new_query or new_query.lower() == current_query.lower():
            print("No useful follow-up retrieval query was produced. Writing with available context.\n")
            break
        current_query = new_query
    else:
        print("Reached the iteration limit. Writing with available context.\n")

    final_prompt = f"""Use the retrieved context below to answer the user's question.

Retrieved context:
{context}

User question:
{query}

Answering requirements:
- Answer the question directly first.
- Ground the answer in the retrieved context.
- Mention specific rules, section names, examples, numbers, or exceptions when available.
- If the context is incomplete, say exactly what is missing and still provide the best supported answer.
- Keep the answer polished and easy to scan."""

    if APPLY_HTML_STYLING:
        final_prompt += """
- IMPORTANT STYLING RULE: You MUST format your response using HTML. Look at the metadata brackets in the retrieved context (e.g., [HTML_TAG: p | CSS_COLOR: rgb(...) | ...]). You must apply the exact CSS properties from the source as inline styles to your output elements. Never use standard markdown bold/italics.
Example format: <p style="color: rgb(0,0,0); font-family: Arial; background-color: transparent;">Your text here.</p>
"""

    answer = generate_answer_with_gpt(final_prompt)
    return answer, all_docs

print("EPO RAG pipeline ready.")
print(f"Model: {LLM_MODEL}")
print(f"Corpus: {INDEX_NAME} | chunks: {len(documents):,}")
print("Ask about the documents.")

while True:
    query = input("\nAsk a question (or 'exit'): ").strip()
    if query.lower() in {"exit", "quit"}:
        print("Done.")
        break
    if not query:
        continue

    answer, sources = rag_answer(query, max_iterations=3)

    print("\n" + "=" * 70)
    print("FINAL ANSWER")
    print("=" * 70 + "\n")

    if APPLY_HTML_STYLING:
        display(HTML(answer))
    else:
        display(Markdown(answer))

    print("\n" + "=" * 70)
    print(f"SOURCES ({len(sources)} reranked chunks)")
    print("=" * 70)

    for index, doc in enumerate(sources[:8], start=1):
        score = doc.metadata.get("rerank_score")
        score_text = f" | score: {score:.4f}" if isinstance(score, float) else ""
        print(f"\n{index}. {source_label(doc, index)}{score_text}")
        print("-" * 70)
        preview = doc.page_content[:350].replace("\n", " ")
        print(f"{preview}...")

    if len(sources) > 8:
        print(f"\n... plus {len(sources) - 8} more chunks")


EPO RAG pipeline ready.
Model: gpt-5.5
Corpus: ingested_data | chunks: 103
Ask about the documents.

Ask a question (or 'exit'): what does the notice number consists of

Question: what does the notice number consists of
Iteration 1/3
Search query: what does the notice number consists of
Retrieved unique chunks: 14 | context chars: 6,140
Context check: The retrieved context includes the sentence 'The notice number consists of:' but only one list item ('the year of publication, comprising four digits') and another possibly unrelated/older item. The full components of the notice number are missing.
Iteration 2/3
Search query: Interinstitutional Style Guide 1.3.2 Notice number consists of year publication four digits C series
Retrieved unique chunks: 22 | context chars: 9,029
Context check: The retrieved context includes the heading and the full list of components of the notice number: the letter C, the four-digit publication year, and a yearly sequential number, plus the format C/YYYY/N.



SOURCES (22 reranked chunks)

1. Source 1 | 1.3.2._Numbering_of_documents_-_Interinstitutional_Style_Guide_-_Publications_Office_of_the_EU_27_7_2026_2_15_12_μ.μ:00004 | /content/drive/MyDrive/Colab Notebooks/RAG/ingested_data/chunks/all_chunks.jsonl | text | score: 0.9890
----------------------------------------------------------------------
The notice number consists of:...

2. Source 2 | 1.3.2._Numbering_of_documents_-_Interinstitutional_Style_Guide_-_Publications_Office_of_the_EU_27_7_2026_2_15_12_μ.μ:00011 | /content/drive/MyDrive/Colab Notebooks/RAG/ingested_data/chunks/all_chunks.jsonl | text | score: 0.9827
----------------------------------------------------------------------
The notice number consisted of:...

3. Source 3 | 1.3.2._Numbering_of_documents_-_Interinstitutional_Style_Guide_-_Publications_Office_of_the_EU_27_7_2026_2_15_12_μ.μ:00002 | /content/drive/MyDrive/Colab Notebooks/RAG/ingested_data/chunks/all_chunks.jsonl | heading | score: 0.9607
------------------------

In [ ]:
import sys

sys.executable

'/opt/homebrew/opt/python@3.10/bin/python3.10'